In [1]:
from datetime import datetime
import pandas as pd
import networkx as nx
from visualize_ean import plot_ean, draw_ean
from visualize_ean_plotly import plot_ean_plotly, draw_ean_plotly, show_ean
from build_ean import build_ean, add_headway_arcs, propagate, enrich_trip_data_with_boundaries
import headway_integration as hi
import numpy as np
from collections import defaultdict
import sys
from pathlib import Path as path

project_root = path(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.schematic_map.routing import (build_route,get_signal_nodes_on_route,)
from infra_data.scenarios import load_network_csv, load_headways, get_scenario
from tools.RailML2trip_data.reassign_trip_id import reassign_trip_ids_by_departure


# Import infrastructure and timetable, clean and prepare data

In [2]:
nodesDf = load_network_csv("nodes.csv", "node_id")
edgesDf = load_network_csv("edges.csv", "edge_id")
scenario = get_scenario()
#edgesDf["length"] = (edgesDf["node_to"].map(nodesDf["pk_rel"])- edgesDf["node_from"].map(nodesDf["pk_rel"])).abs()
#trip_data = np.load(r"C:\Users\LeoC\VSCodes\optimizationVinschgau\plottingRailML\230215 FBS Fpl 2026-NB-1443983-3.npy", allow_pickle=True).item()
trip_data = np.load(rf"C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\tools\RailML2trip_data\trip_data_{scenario}.npy", allow_pickle=True).item()
headway_dict = load_headways()
selected_trips_sorted = np.array(list(trip_data.keys()))

In [3]:
trip_data = {
    train_id: [
        (station, arr_dt, dep_dt, True)
        for station, arr_dt, dep_dt in stops
    ]
    for train_id, stops in trip_data.items()
}

In [4]:
trip_data, ordered_old_ids=reassign_trip_ids_by_departure(trip_data)

In [5]:
routing_results = {}

for stop_col in ["stop_slow", "stop_fast"]:

    stops = nodesDf.index[(nodesDf[stop_col] == 1) & (nodesDf["y"] == 0)].tolist()

    for start, end in zip(stops[:-1], stops[1:]):

        routing_results[(start, end)] = build_route(nodesDf, edgesDf, start, end)
        routing_results[(end, start)] = build_route(nodesDf, edgesDf, end, start)

In [6]:
routes = {}
for trip_id, trip in trip_data.items():
    start, end = trip[0][0], trip[-1][0]
    routes[trip_id] = build_route(nodesDf, edgesDf, start, end)

In [7]:
#add side nodes to trip data

from copy import deepcopy

def add_side_nodes_to_trip_data(trip_data, nodesDf):
    """
    Return a copy of trip_data where terminal nodes are replaced by their
    '_side' counterpart, but only for trains running from Merano to Malles
    (i.e. decreasing pk_rel).

    A stop 'X' is replaced by 'X_side' only if:
      - 'X_side' exists in nodesDf, and
      - the train direction is Merano -> Malles.
    """
    trip_data_sides = deepcopy(trip_data)

    available_nodes = set(nodesDf.index)

    for train_id, stops in trip_data_sides.items():

        # Need at least two stops to infer direction
        if len(stops) < 2:
            continue

        pk0 = nodesDf.loc[stops[0][0], "pk_rel"]
        pk1 = nodesDf.loc[stops[1][0], "pk_rel"]

        # Merano -> Malles corresponds to decreasing pk_rel
        merano_to_malles = pk1 > pk0

        if not merano_to_malles:
            continue

        for i, (node, arr, dep, is_stop) in enumerate(stops):

            side_node = f"{node}_side"

            if side_node in available_nodes:
                stops[i] = (side_node, arr, dep, True)

    return trip_data_sides

In [8]:
trip_data_sides = add_side_nodes_to_trip_data(trip_data, nodesDf)

In [9]:
IG = hi.build_infra_graph(edgesDf)
chains, boundary_nodes = hi.extract_chains(IG, nodesDf)

print("Chains:", chains)
print("Boundary Nodes:", boundary_nodes)
trip_data_enriched = enrich_trip_data_with_boundaries(trip_data_sides, routes, nodesDf, boundary_nodes)

Chains: {('Dev_TEL_01', 'Dev_TEL_02'): ['Dev_TEL_01', 'Pa_I_TEL_31b_side', 'Pr_I_TEL_51_side', 'Pa_I_TEL_31a_side', 'TEL_side', 'Pa_I_TEL_32_side', 'Dev_TEL_02'], ('Dev_TEL_02', 'Dev_STA_01'): ['Dev_TEL_02', 'Pr_I_TEL_52', 'Pa_E_TEL_62', 'Pr_E_TEL_02', 'Pr_E_RAB_01', 'Pr_I_RAB_51', 'PL08', 'Pa_I_RAB_31', 'RAB', 'Pa_RAB_32', 'Pr_E_RAB_02', 'HD_302_02', 'Pr_E_PLA_01', 'Dev_01_PLA', 'Pa_PLA_31', 'PLA', 'Pa_PLA_32', 'PL09', 'Pr_I_PLA_52', 'Pr_E_PLA_02', 'HD_303_02', 'HD_303_04', 'km48500', 'Pr_E_NAT_01', 'Pr_I_NAT_51', 'PL10', 'Dev_NAT_01', 'Pa_NAT_31', 'NAT', 'Pa_NAT_32', 'Dev_NAT_02', 'Pr_E_NAT_02', 'Pr_E_PL11_01', 'Pr_I_PL11_51', 'PL11', 'Pr_I_PL11_52', 'Dev_PL11_02', 'Pr_E_PL11_02', 'SblNatKomp', 'Pr_E_STA_01', 'Pa_E_STA_61', 'Pr_I_STA_51', 'Dev_STA_01'], ('Dev_MAL_05', 'MAL'): ['Dev_MAL_05', 'Pa_I_MAL_31d', 'MAL'], ('Dev_SIL_05', 'Dev_SIL_02'): ['Dev_SIL_05', 'Pa_I_SIL_31_side', 'SIL_side', 'Pa_I_SIL_32_side', 'Dev_SIL_02'], ('Dev_SIL_02', 'Dev_LASA_01'): ['Dev_SIL_02', 'Pr_I_SIL_52',

In [10]:
# Remove opposite-direction headways on double-track chains
headway_dict = deepcopy(headway_dict)

# --------------------------------------------------
# Step 1: Count how many infrastructure edges occupy
# each elementary pk interval
# --------------------------------------------------

pk_values = sorted(nodesDf["pk_rel"].unique())

interval_count = defaultdict(int)

for _, edge in edgesDf.iterrows():

    pk1 = nodesDf.loc[edge["node_from"], "pk_rel"]
    pk2 = nodesDf.loc[edge["node_to"], "pk_rel"]

    a, b = sorted((pk1, pk2))

    for left, right in zip(pk_values[:-1], pk_values[1:]):
        if left >= a and right <= b:
            interval_count[(left, right)] += 1


# --------------------------------------------------
# Step 2: Determine whether each chain is entirely
# on double track
# --------------------------------------------------

double_track_chains = set()

for chain_key, chain_nodes in chains.items():

    is_double = True

    for n1, n2 in zip(chain_nodes[:-1], chain_nodes[1:]):

        pk1 = nodesDf.loc[n1, "pk_rel"]
        pk2 = nodesDf.loc[n2, "pk_rel"]

        a, b = sorted((pk1, pk2))

        for left, right in zip(pk_values[:-1], pk_values[1:]):
            if left >= a and right <= b:
                if interval_count[(left, right)] < 2:
                    is_double = False
                    break

        if not is_double:
            break

    if is_double:
        double_track_chains.add(chain_key)


# --------------------------------------------------
# Step 3: Remove opposite-direction headways
# on double-track chains
# --------------------------------------------------

for key in list(headway_dict):

    chain_key, cat1, cat2 = key

    if chain_key in double_track_chains:

        opposite = (
            ("up" in cat1 and "down" in cat2)
            or
            ("down" in cat1 and "up" in cat2)
        )

        if opposite:
            del headway_dict[key]

# Build EAN, add headway constraints

In [11]:
constraints, skipped = hi.assemble_headway_constraints(trip_data_sides, trip_data_enriched, routes, nodesDf, chains, headway_dict)


print("Constraints:", constraints)
print("Skipped:", skipped)

G_scheduled = build_ean(trip_data_enriched)
G_scheduled = add_headway_arcs(G_scheduled, constraints)
assert nx.is_directed_acyclic_graph(G_scheduled), "graph must stay a DAG"

Chain ('Dev_LAC_01', 'Dev_LAC_02'): 5 up trains, 5 down trains
Assembling same-direction constraints for chain ('Dev_LAC_01', 'Dev_LAC_02')
1 2
2 3
3 4
4 5
6 7
7 8
8 9
9 10
Assembling opposite-direction constraints for chain ('Dev_LAC_01', 'Dev_LAC_02')
1 6
1 7
1 8
1 9
1 10
2 6
2 7
2 8
2 9
2 10
3 6
3 7
3 8
3 9
3 10
4 6
4 7
4 8
4 9
4 10
5 6
5 7
5 8
5 9
5 10
Chain ('Dev_STA_02', 'Dev_LAC_01'): 5 up trains, 5 down trains
Assembling same-direction constraints for chain ('Dev_STA_02', 'Dev_LAC_01')
1 2
2 3
3 4
4 5
6 7
7 8
8 9
9 10
Assembling opposite-direction constraints for chain ('Dev_STA_02', 'Dev_LAC_01')
1 6
1 7
1 8
1 9
1 10
2 6
2 7
2 8
2 9
2 10
3 6
3 7
3 8
3 9
3 10
4 6
4 7
4 8
4 9
4 10
5 6
5 7
5 8
5 9
5 10
Chain ('Dev_STA_01', 'Dev_STA_02'): 5 up trains, 5 down trains
Assembling same-direction constraints for chain ('Dev_STA_01', 'Dev_STA_02')
1 2
2 3
3 4
4 5
6 7
7 8
8 9
9 10
Assembling opposite-direction constraints for chain ('Dev_STA_01', 'Dev_STA_02')
1 6
1 7
1 8
1 9
1 10
2 6
2 7

In [12]:
station = "Dev_STA_02"
edges = [(u, v) for u, v in G_scheduled.edges if station in u or station in v]
edges

[((1, 'STA_side', 'dep', 13), (1, 'Dev_STA_02', 'arr', 14)),
 ((1, 'Dev_STA_02', 'arr', 14), (1, 'Dev_STA_02', 'dep', 14)),
 ((1, 'Dev_STA_02', 'dep', 14), (1, 'CIA', 'arr', 15)),
 ((1, 'Dev_STA_02', 'dep', 14), (2, 'Dev_STA_02', 'dep', 14)),
 ((2, 'STA_side', 'dep', 13), (2, 'Dev_STA_02', 'arr', 14)),
 ((2, 'Dev_STA_02', 'arr', 14), (2, 'Dev_STA_02', 'dep', 14)),
 ((2, 'Dev_STA_02', 'dep', 14), (2, 'CIA', 'arr', 15)),
 ((2, 'Dev_STA_02', 'dep', 14), (3, 'Dev_STA_02', 'dep', 14)),
 ((3, 'STA_side', 'dep', 13), (3, 'Dev_STA_02', 'arr', 14)),
 ((3, 'Dev_STA_02', 'arr', 14), (3, 'Dev_STA_02', 'dep', 14)),
 ((3, 'Dev_STA_02', 'dep', 14), (3, 'CIA', 'arr', 15)),
 ((3, 'Dev_STA_02', 'dep', 14), (4, 'Dev_STA_02', 'dep', 14)),
 ((4, 'STA_side', 'dep', 13), (4, 'Dev_STA_02', 'arr', 14)),
 ((4, 'Dev_STA_02', 'arr', 14), (4, 'Dev_STA_02', 'dep', 14)),
 ((4, 'Dev_STA_02', 'dep', 14), (4, 'CIA', 'arr', 15)),
 ((4, 'Dev_STA_02', 'dep', 14), (5, 'Dev_STA_02', 'dep', 14)),
 ((5, 'STA_side', 'dep', 13)

In [13]:
u = (1, "Dev_LAC_01", "arr", 17)
v = (6, "Dev_LAC_01", "dep", 16)

if G_scheduled.has_edge(u, v):
    print(u, v, G_scheduled.get_edge_data(u, v))

(1, 'Dev_LAC_01', 'arr', 17) (6, 'Dev_LAC_01', 'dep', 16) {'scheduled_duration': 388.2659849999982, 'min_duration': 96.58354005653393, 'kind': 'headway', 'train_i': 1, 'train_j': 6, 'resource': ('Dev_STA_02', 'Dev_LAC_01'), 'is_active': False}


In [14]:
bad = []

def time_to_sec(t):
        return t.hour * 3600 + t.minute * 60 + t.second

def event_time_sec(train_id, seq, event):
        t = trip_data_enriched[train_id][seq][2 if event == "dep" else 1]
        return time_to_sec(t)

for c in constraints:
    ti = event_time_sec(c["train_i"], c["seq_i"], c["event_i"])
    tj = event_time_sec(c["train_j"], c["seq_j"], c["event_j"])
    if ti > tj:
        bad.append((c, ti, tj))

print(len(bad))

0


# Stochastic Perturbation

In [15]:
node_perturbations, edge_perturbations = hi.generate_perturbation_scenarios(G_scheduled,
    n_scenarios=1,
    entry_delay_mean=1*60,
    entry_delay_std=0.25*60,
    running_delay_mean=0.25*60,
    running_delay_std=0.25*60,
    perturbation_probability = 0.2,
    seed=42,
)

## Propagation

### Plotly

In [16]:
realized_graphs = []

for node_p, edge_p in zip(node_perturbations,edge_perturbations,):

    G_realized = propagate(G_scheduled,edge_perturbations=edge_p,node_perturbations=node_p,)
    realized_graphs.append(G_realized)


fig, ax = plot_ean_plotly(G_scheduled,nodesDf,edgesDf,title="Scheduled vs realized",)


for G_realized in realized_graphs:

    draw_ean_plotly(G_realized,nodesDf,ax,alpha=0.5,linewidth_scale=0.8,)


show_ean(fig,filename=f"ean_visualization{scenario}.html",auto_open=True,)

EAN visualization written to: C:\Users\LeoC\VSCodes\optimizationVinschgau\AusbauVinschgau\ean_simulation\ean_visualization0.html


WindowsPath('ean_visualization0.html')

In [17]:
G_scheduled.nodes[(2, "Dev_ME_01", "dep", 0)]["seq"]

KeyError: (2, 'Dev_ME_01', 'dep', 0)

### Matplotlib

In [ ]:
'''realized_graphs = []

for node_p, edge_p in zip(node_perturbations, edge_perturbations):

    G_realized = propagate2(
        G_scheduled,
        edge_perturbations=edge_p,
    )

    realized_graphs.append(G_realized)


fig, ax = plot_ean(G_scheduled,nodesDf,edgesDf,title="Scheduled vs realized")

for G_realized in realized_graphs:
    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)'''

'realized_graphs = []\n\nfor node_p, edge_p in zip(node_perturbations, edge_perturbations):\n\n    G_realized = propagate2(\n        G_scheduled,\n        edge_perturbations=edge_p,\n    )\n\n    realized_graphs.append(G_realized)\n\n\nfig, ax = plot_ean(G_scheduled,nodesDf,edgesDf,title="Scheduled vs realized")\n\nfor G_realized in realized_graphs:\n    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)'

## Report

In [ ]:
def format_time(seconds):
    seconds = round(seconds)
    sign = "-" if seconds < 0 else ""
    seconds = abs(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{sign}{hours:02d}:{minutes:02d}:{seconds:02d}"


def get_train_times(G):
    """Returns {train_id: {"dep": ..., "arr": ...}}."""
    times = {}
    trains = sorted({data["train"] for _, data in G.nodes(data=True) if "train" in data})

    for train in trains:
        events = [data for _, data in G.nodes(data=True) if data.get("train") == train]
        times[train] = {
            "dep": min(e["time"] for e in events if e["event"] == "dep"),
            "arr": max(e["time"] for e in events if e["event"] == "arr")
        }

    return times


def compare_to_schedule(G_scheduled, G_realized):
    scheduled = get_train_times(G_scheduled)
    realized = get_train_times(G_realized)

    report = {}

    for train in scheduled:
        report[train] = {
            "dep_sched": scheduled[train]["dep"],
            "dep_real": realized[train]["dep"],
            "arr_sched": scheduled[train]["arr"],
            "arr_real": realized[train]["arr"],
            "delay": realized[train]["arr"] - scheduled[train]["arr"]
        }

    return report


for i, G_realized in enumerate(realized_graphs):
    
    arrival_report = compare_to_schedule(G_scheduled, G_realized)

    print(f"\nScenario {i}")
    print("-" * 100)

    for train in sorted(arrival_report):
        r = arrival_report[train]

        print(
            f"Train {train:>2}: "
            f"dep_sched={format_time(r['dep_sched'])}   "
            f"dep_real={format_time(r['dep_real'])}   "
            f"arr_sched={format_time(r['arr_sched'])}   "
            f"arr_real={format_time(r['arr_real'])}   "
            f"delay={format_time(r['delay'])}"
        )


Scenario 0
----------------------------------------------------------------------------------------------------
Train  1: dep_sched=16:00:03   dep_real=16:01:08   arr_sched=17:26:12   arr_real=17:25:55   delay=-00:00:17
Train  2: dep_sched=17:00:03   dep_real=17:00:47   arr_sched=18:26:12   arr_real=18:26:09   delay=-00:00:03
Train  3: dep_sched=18:00:03   dep_real=18:01:14   arr_sched=19:26:12   arr_real=19:26:47   delay=00:00:35
Train  4: dep_sched=19:00:03   dep_real=19:01:17   arr_sched=20:26:12   arr_real=20:25:55   delay=-00:00:17
Train  5: dep_sched=20:00:03   dep_real=20:00:34   arr_sched=21:26:12   arr_real=21:26:37   delay=00:00:25
Train  6: dep_sched=16:05:57   dep_real=16:06:37   arr_sched=17:31:45   arr_real=17:31:37   delay=-00:00:08
Train  7: dep_sched=17:05:57   dep_real=17:06:59   arr_sched=18:31:45   arr_real=18:31:49   delay=00:00:04
Train  8: dep_sched=18:05:57   dep_real=18:06:52   arr_sched=19:31:45   arr_real=19:31:37   delay=-00:00:08
Train  9: dep_sched=19:05:

In [ ]:
stop

NameError: name 'stop' is not defined

# Manual Perturbation

## Propagation

In [ ]:
#inject delays manually
running_edges = [(u, v) for u, v, data in G_scheduled.edges(data=True) if data["kind"] == "running"]
edge = running_edges[0]  

#perturbations = [{},{edge: 60},{edge: 120},{edge: 300}]
perturbations = [{edge:0}]


realized_graphs = []

for p in perturbations:

    G_realized = propagate(G_scheduled, p)

    realized_graphs.append(G_realized)


fig, ax = plot_ean(G_scheduled, nodesDf, edgesDf, title="Scheduled vs realized")

for G_realized in realized_graphs:
    draw_ean(G_realized,nodesDf,ax,alpha=0.2,linewidth_scale=0.8,)

## Report

In [ ]:
for i, (node_perturbation, edge_perturbation, G_realized) in enumerate(
    zip(node_perturbations, edge_perturbations, realized_graphs)
):
    arrival_report = compare_to_schedule(G_scheduled, G_realized)

    print(f"\nScenario {i}")
    print("-" * 100)

    for train in sorted(arrival_report):
        r = arrival_report[train]

        print(
            f"Train {train:>2}: "
            f"dep_sched={format_time(r['dep_sched'])}   "
            f"dep_real={format_time(r['dep_real'])}   "
            f"arr_sched={format_time(r['arr_sched'])}   "
            f"arr_real={format_time(r['arr_real'])}   "
            f"delay={format_time(r['delay'])}"
        )